<a href="https://colab.research.google.com/github/koewilliams5/DI-Bootcamp/blob/main/mcp_gemini_workspace_assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mini-Projet : Integration MCP + Agent IA avec Gemini

**Theme choisi : "Workspace assistant"** (assistant d'espace de travail)

L'agent va pouvoir combiner trois outils, exposes via le protocole **MCP** (Model
Context Protocol), pour aider un developpeur/etudiant a naviguer dans un petit
espace de travail :

1. **`filesystem`** (serveur MCP tiers, officiel) : lire/lister des fichiers.
2. **`git`** (serveur MCP tiers, officiel) : consulter l'historique d'un depot git.
3. **`custom_ops`** (serveur MCP maison, ecrit avec FastMCP) : un outil de resume
   de texte + un outil specifique au theme qui compte les taches faites/a faire
   dans une checklist markdown.

**Point cle de l'exercice :** ce n'est PAS un pipeline code en dur (if/else). C'est
**Gemini** (le LLM) qui, a chaque etape, decide lui-meme quel outil appeler (ou
aucun), en se basant sur la question posee et sur la description des outils
disponibles. C'est ca, un agent "agentique" : la politique de decision est
confiee au modele, pas ecrite a la main par nous.

Environnement cible : **Google Colab** (le notebook peut aussi tourner en local
si Node/NPM et Python sont deja installes).


## 0) Un peu de theorie : MCP et agent LLM

- **MCP (Model Context Protocol)** est un protocole standard qui permet a un
  agent (le "client MCP") de decouvrir et d'appeler des outils exposes par des
  **serveurs MCP** independants, sans avoir a coder une integration specifique
  pour chacun.
- Chaque serveur MCP tourne comme un **sous-processus** (ici, via `stdio` :
  l'agent communique avec le serveur via son entree/sortie standard, comme s'il
  discutait avec un programme en ligne de commande).
- `langchain-mcp-adapters` fait le pont entre ces serveurs MCP et les outils
  (`Tool`) que LangChain/LangGraph sait utiliser nativement.
- L'agent (construit avec LangGraph) recoit : (a) le modele Gemini, (b) la
  liste des outils MCP disponibles. A chaque tour, Gemini lit la question de
  l'utilisateur + les descriptions des outils, et decide s'il doit appeler un
  outil, lequel, avec quels arguments, ou s'il peut repondre directement.


## 1) Installation des dependances

In [ ]:
# Dependances principales : LangChain/LangGraph, l'integration Gemini,
# le client MCP pour LangChain, et nest_asyncio (necessaire pour faire tourner
# des boucles asyncio imbriquees dans un notebook Colab/Jupyter).
%pip install -qU \
  "langchain>=0.3" \
  "langgraph>=0.2" \
  "langchain-google-genai>=2.0" \
  "google-genai>=1.0" \
  "langchain-mcp-adapters==0.2.1" \
  "nest_asyncio"


## 2) Configuration de la cle API Gemini (`GOOGLE_API_KEY`)

In [ ]:
import os
import getpass

# On demande la cle de maniere securisee (elle ne s'affiche pas a l'ecran et
# n'est jamais ecrite en dur dans le notebook).
if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Entrez votre GOOGLE_API_KEY : ")

print("GOOGLE_API_KEY configuree :", bool(os.environ.get("GOOGLE_API_KEY")))

# Alternative dans Colab : si vous avez enregistre la cle dans les "Secrets"
# du notebook (icone cle a gauche), vous pouvez plutot faire :
#
#   from google.colab import userdata
#   os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")


## 3) Verification de Node/NPM (necessaire pour les serveurs MCP lances via `npx`)

In [ ]:
!node --version
!npx --version


In [ ]:
# Si la cellule precedente a echoue (Node/NPM absents), on les installe.
!apt-get -qq update
!apt-get -qq install -y nodejs npm
!node --version
!npx --version


## 4) Preparation de l'espace de travail (`WORKDIR`)

Avant de brancher les serveurs MCP `filesystem` et `git`, on prepare un petit
dossier de travail avec quelques fichiers et un vrai depot git (sinon les
outils n'auront rien a inspecter).


In [ ]:
import subprocess
from pathlib import Path

WORKDIR = "/content/workspace"
Path(WORKDIR).mkdir(parents=True, exist_ok=True)

# Quelques fichiers d'exemple pour donner du contenu au serveur `filesystem`.
(Path(WORKDIR) / "README.md").write_text(
    "# Workspace Assistant\n\nCeci est un espace de travail de demonstration pour le mini-projet MCP + Gemini.\n",
    encoding="utf-8",
)
(Path(WORKDIR) / "notes.md").write_text(
    "Notes de reunion :\n- Discussion sur l'architecture de l'agent.\n- A revoir : gestion des erreurs MCP.\n",
    encoding="utf-8",
)
# Une checklist markdown : sera utilisee plus tard par notre outil MCP maison
# `count_todo_items` (voir section 8).
(Path(WORKDIR) / "todo.md").write_text(
    "- [x] Installer les dependances\n"
    "- [x] Configurer la cle API Gemini\n"
    "- [ ] Connecter les serveurs MCP\n"
    "- [ ] Ecrire le serveur MCP personnalise\n"
    "- [ ] Tester l'agent complet\n",
    encoding="utf-8",
)

# On initialise un vrai depot git dans WORKDIR pour que le serveur MCP `git`
# ait un historique reel a interroger (git log, git status, etc.).
subprocess.run(["git", "init"], cwd=WORKDIR, check=True)
subprocess.run(["git", "config", "user.email", "student@example.com"], cwd=WORKDIR, check=True)
subprocess.run(["git", "config", "user.name", "Student"], cwd=WORKDIR, check=True)
subprocess.run(["git", "add", "."], cwd=WORKDIR, check=True)
subprocess.run(["git", "commit", "-m", "Initial commit: setup workspace"], cwd=WORKDIR, check=True)

print("WORKDIR pret :", WORKDIR)
print("Contenu :", [p.name for p in Path(WORKDIR).iterdir()])


## 5) Choix des serveurs MCP tiers : `filesystem` et `git`

Ce sont deux serveurs MCP **officiels** (maintenus par le projet
Model Context Protocol) :

- **`@modelcontextprotocol/server-filesystem`** (paquet npm, lance via `npx`) :
  expose des outils pour lire/lister des fichiers dans un dossier autorise.
- **`mcp-server-git`** (paquet Python) : expose des outils pour interroger
  l'historique et le statut d'un depot git.


In [ ]:
# Le serveur git est distribue en tant que paquet Python : on l'installe.
%pip install -qU mcp-server-git


In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

# Chaque entree decrit COMMENT lancer le serveur MCP en sous-processus (stdio) :
# - "command"/"args" : la commande shell a executer (ex: npx ... ou python -m ...)
# - "transport": "stdio" : l'agent communique avec le serveur via son entree/sortie standard
mcp_connections = {
    "filesystem": {
        "transport": "stdio",
        "command": "npx",
        "args": ["-y", "@modelcontextprotocol/server-filesystem", WORKDIR],
    },
    "git": {
        "transport": "stdio",
        "command": "python",
        "args": ["-m", "mcp_server_git", "--repository", WORKDIR],
    },
}

print("Serveurs MCP configures :", list(mcp_connections.keys()))


## 6) Connexion aux serveurs MCP depuis l'agent

`MultiServerMCPClient` lance les sous-processus definis ci-dessus et recupere,
pour chaque serveur, la liste des outils qu'il expose (avec leur nom, leur
description et leur schema d'arguments). `tool_name_prefix=True` prefixe
chaque nom d'outil par le nom du serveur (ex: `filesystem__read_file`) pour
eviter toute collision de noms entre serveurs.


In [ ]:
import asyncio
import nest_asyncio

# Necessaire dans Colab/Jupyter : permet d'executer une boucle asyncio "imbriquee"
# (le notebook a deja sa propre boucle asyncio en arriere-plan).
nest_asyncio.apply()

client = MultiServerMCPClient(mcp_connections, tool_name_prefix=True)

# get_tools() est une coroutine : on la lance de maniere synchrone avec run_until_complete.
tools = asyncio.get_event_loop().run_until_complete(client.get_tools())

print("Nombre d'outils recuperes :", len(tools))
for t in tools:
    print("-", t.name, ":", t.description[:80])


## 7) Construction d'un agent Gemini avec LangGraph

On utilise `create_react_agent` (LangGraph) : cette fonction construit un
agent de type "ReAct" (Reasoning + Acting), c'est-a-dire une boucle ou le
modele peut, a chaque etape, soit appeler un outil, soit produire une reponse
finale. C'est cette boucle qui rend l'agent "agentique" : la sequence
d'actions n'est pas fixee a l'avance, elle emerge des decisions du modele.


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.prebuilt import create_react_agent

# temperature=0 : reponses plus deterministes, utiles pour un exercice ou l'on
# veut observer un comportement stable de l'agent.
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0)

SYSTEM_PROMPT = (
    "Tu es un assistant d'espace de travail (workspace assistant). "
    "Tu as acces a des outils pour lire des fichiers, consulter l'historique git, "
    "et resumer du texte. "
    "Choisis toujours l'outil le plus pertinent pour la question posee ; "
    "si aucun outil n'est necessaire, reponds directement. "
    "Sois concis (2 a 4 phrases) et mentionne, si pertinent, le nom du fichier "
    "ou de l'outil utilise pour obtenir l'information."
)

# NB : selon la version de langgraph installee, le parametre du prompt systeme
# peut s'appeler `prompt=` (versions recentes) ou `state_modifier=` (versions
# plus anciennes). Si vous obtenez une TypeError ici, essayez de remplacer
# `prompt=SYSTEM_PROMPT` par `state_modifier=SYSTEM_PROMPT`.
agent = create_react_agent(llm, tools, prompt=SYSTEM_PROMPT)

print("Agent Gemini pret avec", len(tools), "outils.")


In [ ]:
def ask(agent_to_use, question: str):
    """
    Envoie une question a l'agent et affiche :
      - les appels d'outils effectues (nom + arguments), dans l'ordre
      - la reponse finale du modele

    C'est ce qui permet d'INSPECTER le comportement agentique (quels outils
    ont ete choisis, avec quels parametres) plutot que de juste voir le
    resultat final.
    """
    result = asyncio.get_event_loop().run_until_complete(
        agent_to_use.ainvoke({"messages": [{"role": "user", "content": question}]})
    )
    messages = result["messages"]

    print("Question :", question)
    for m in messages:
        tool_calls = getattr(m, "tool_calls", None)
        if tool_calls:
            for tc in tool_calls:
                print(f"  -> appel outil : {tc['name']}({tc['args']})")

    final_answer = messages[-1].content
    print("Reponse :", final_answer)
    print("-" * 60)
    return final_answer


## 8) Premier test rapide (`filesystem` + `git`)

On verifie que l'agent sait deja utiliser les deux serveurs tiers, avant
d'ajouter notre outil personnalise.


In [ ]:
ask(agent, "Liste les fichiers presents dans l'espace de travail.")
ask(agent, "Que dit le dernier commit git de ce depot ?")


## 9) Implementation d'un serveur MCP personnalise avec FastMCP

On ecrit notre propre serveur MCP (`custom_ops`), qui expose :
- `ping` : un outil de sante simple, pour verifier que le serveur repond.
- `summarize_lines` : un outil generique qui calcule des statistiques sur une
  liste de lignes de texte.
- `count_todo_items` : un outil specifique au theme "workspace assistant" qui
  lit une checklist markdown (`- [ ]` / `- [x]`) et compte les taches faites
  et a faire.


In [ ]:
%pip install -qU "fastmcp>=2.0.0"


In [ ]:
from pathlib import Path

server_lines = ['from pathlib import Path', 'from typing import Dict, List', '', 'from fastmcp import FastMCP', '', 'mcp = FastMCP(name="custom_ops")', '', '', '@mcp.tool', 'def ping() -> str:', "    '''Outil de sante : verifie que le serveur MCP personnalise repond bien.'''", '    return "pong"', '', '', '@mcp.tool', 'def summarize_lines(lines: List[str]) -> Dict[str, int]:', "    '''Outil generique : renvoie des statistiques sur une liste de lignes de texte", "    (nombre total de lignes, nombre de lignes non vides).'''", '    total = len(lines)', '    nonempty = sum(1 for l in lines if l.strip())', '    return {"total_lines": total, "nonempty_lines": nonempty}', '', '', '@mcp.tool', 'def count_todo_items(path: str) -> Dict[str, int]:', "    '''Outil specifique au theme workspace assistant : lit un fichier de type", "    checklist markdown (lignes '- [ ] ...' ou '- [x] ...') et renvoie le nombre", "    de taches faites, a faire, et le total.'''", '    text = Path(path).read_text(encoding="utf-8")', '    lines = text.splitlines()', '    done = sum(1 for l in lines if l.strip().lower().startswith("- [x]"))', '    todo = sum(1 for l in lines if l.strip().lower().startswith("- [ ]"))', '    return {"done": done, "todo": todo, "total": done + todo}', '', '', 'if __name__ == "__main__":', '    mcp.run(transport="stdio")', '']

server_code = chr(10).join(server_lines)

server_path = Path("/content/custom_mcp_server.py")
server_path.write_text(server_code, encoding="utf-8")

print("Fichier ecrit :", server_path)
print(server_code)


## 10) Ajout du serveur personnalise au client MCP (fusion des outils)

On ajoute `custom_ops` a `mcp_connections`, puis on cree un nouveau client MCP
qui va lancer les 3 serveurs (`filesystem`, `git`, `custom_ops`) et recuperer
l'ensemble de leurs outils.


In [ ]:
mcp_connections["custom_ops"] = {
    "transport": "stdio",
    "command": "python",
    "args": [str(server_path)],
}

client2 = MultiServerMCPClient(mcp_connections, tool_name_prefix=True)
tools2 = asyncio.get_event_loop().run_until_complete(client2.get_tools())

print("Nombre total d'outils :", len(tools2))
print("Outils custom_ops :", [t.name for t in tools2 if "custom_ops" in t.name])


## 11) Reconstruction de l'agent avec l'ensemble des outils

On reconstruit l'agent avec `tools2` (filesystem + git + custom_ops), en
mettant a jour le prompt systeme pour qu'il mentionne le nouvel outil.


In [ ]:
SYSTEM_PROMPT_V2 = (
    "Tu es un assistant d'espace de travail (workspace assistant). "
    "Tu as acces a des outils pour : lire des fichiers, consulter l'historique git, "
    "resumer une liste de lignes de texte (summarize_lines), et compter les taches "
    "faites/a faire dans une checklist markdown (count_todo_items). "
    "Choisis toujours l'outil le plus pertinent pour la question posee ; "
    "si aucun outil n'est necessaire, reponds directement. "
    "Sois concis (2 a 4 phrases) et mentionne, si pertinent, le nom du fichier "
    "ou de l'outil utilise pour obtenir l'information. "
    "Si tu ne trouves aucune information pertinente avec les outils disponibles, "
    "dis-le explicitement plutot que d'inventer une reponse."
)

agent_v2 = create_react_agent(llm, tools2, prompt=SYSTEM_PROMPT_V2)

print("Agent Gemini (v2) pret avec", len(tools2), "outils.")


## 12) Verification rapide : 3 requetes types

Une requete par outil, pour bien observer quel outil l'agent choisit et
pourquoi :
1. Une requete qui necessite `filesystem` (lire un fichier).
2. Une requete qui necessite `git` (historique du depot).
3. Une requete qui necessite l'outil personnalise `count_todo_items`.


In [ ]:
test_questions = [
    "Que contient le fichier README.md de l'espace de travail ?",
    "Combien y a-t-il de commits dans le depot git ?",
    "Combien de taches sont deja terminees dans le fichier todo.md, et combien reste-t-il a faire ?",
]

for q in test_questions:
    ask(agent_v2, q)


## 13) Conclusion et pistes d'amelioration

- L'architecture reste **la meme** que celle vue en cours : un client MCP
  regroupe des outils venant de plusieurs serveurs independants (deux
  officiels + un maison), et c'est le LLM (Gemini) qui decide, a l'execution,
  quel outil appeler pour chaque question — rien n'est code en dur.
- Ajouter un nouveau serveur MCP (tiers ou personnalise) ne demande **aucune
  modification** du code de l'agent lui-meme : il suffit de l'ajouter a
  `mcp_connections` et de reconstruire la liste d'outils. C'est le principal
  interet de MCP : la composition de capacites reste modulaire.

Pistes pour aller plus loin (non demandees dans cet exercice) :
- Ajouter un outil de recherche web ou un outil de formatage Markdown pour
  enrichir le theme "workspace assistant".
- Logguer systematiquement les appels d'outils (nom, arguments, duree) dans un
  fichier, pour analyser le comportement de l'agent sur davantage de
  questions.
- Ajouter une gestion d'erreur explicite si un serveur MCP ne demarre pas
  (ex: Node/NPM manquant, mauvais chemin de depot git).
